# Simple Web RAG — Notebook

RAG means Retrieval-Augmented Generation. This notebook reads a public web page and answers questions about it using Gemini.

**Pipeline:**
1. **Load** — fetch the page and turn it into a LangChain `Document`
2. **Split** — break the page into chunks
3. **Embed + Store** — Gemini embeddings turn each chunk into a vector, stored in Chroma
4. **Retrieve + Answer** — a retrieval chain finds the best chunks and asks Gemini for a **structured** (Pydantic) answer, not just plain text
5. **Inspect** — see exactly which chunks were used as evidence

## 0. Imports & Configuration

Everything the pipeline needs: `dompruner` to load the page, `RecursiveCharacterTextSplitter` to chunk it, `GoogleGenerativeAIEmbeddings`/`Chroma` to embed and store it, and `ChatGoogleGenerativeAI` + the retrieval chain helpers to answer from it.

In [27]:
import os
from pathlib import Path

import dompruner
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# .env lives at the repo root, one level above this notebook.
load_dotenv(Path("..") / ".env", override=True)

DEFAULT_CHUNK_SIZE = 1000
DEFAULT_CHUNK_OVERLAP = 100
DEFAULT_TOP_K = 3
EMBEDDING_MODEL = "models/gemini-embedding-001"
CHAT_MODEL = "gemini-2.5-flash"
FALLBACK_ANSWER = "I don't have knowledge about that based on the provided document."

## 1. Validate the Gemini API Key

A small helper that checks the key is present and raises a clear error if it's missing.

In [28]:
def validate_environment() -> str:
    """Return the Gemini API key or raise a clear error."""
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        raise EnvironmentError(
            "GEMINI_API_KEY was not found. Add it to the repo-root .env file."
        )
    return api_key

## 2. Confirm the Key Loaded

In [29]:
validate_environment()
print("GEMINI_API_KEY loaded.")

GEMINI_API_KEY loaded.


## 3. Load the Web Page

`dompruner` fetches the URL and prunes the DOM down to clean Markdown (no headless browser, no API key). We wrap the result in a LangChain `Document`.

In [30]:
URL = "https://www.ashishsubedi.com.np/"

page = dompruner.sync_run(dompruner.run_pipeline(URL))

document = Document(
    page_content=page.markdown,
    metadata={"source": URL, "title": page.meta.get("title")},
)
raw_documents = [document]

document_count = len(raw_documents)
print("Documents loaded:", document_count)
print("Source URL:", URL)
print()
print("Page content preview:")
print(raw_documents[0].page_content[:500])

Documents loaded: 1
Source URL: https://www.ashishsubedi.com.np/

Page content preview:
## About Me

# Ashish Subedi

I'm a fullstack developer from Nepal who builds end-to-end web applications — from pixel-perfect UIs to robust backend APIs. I work across the full stack with React , Next.js , NestJS , and TypeScript , backed by PostgreSQL for reliable, scalable data storage.

I've shipped products at startups and as a freelancer — designing schemas, building REST APIs, and crafting fast, accessible frontends that users love. I care as much about clean architecture and database per


## 4. Split into Chunks

`RecursiveCharacterTextSplitter` breaks each document into overlapping chunks that are small enough to embed and retrieve accurately, while preserving the source metadata on every chunk.

In [31]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=DEFAULT_CHUNK_SIZE,
    chunk_overlap=DEFAULT_CHUNK_OVERLAP,
)
chunks = splitter.split_documents(raw_documents)

for index, chunk in enumerate(chunks, start=1):
    chunk.metadata["chunk_number"] = index

chunk_count = len(chunks)
print("Chunks created:", chunk_count)
print("Max characters per chunk:", DEFAULT_CHUNK_SIZE)
print()

if chunks:
    first_chunk_preview = chunks[0].page_content[:300]
else:
    first_chunk_preview = "(no chunks)"

print("First chunk preview:")
print(first_chunk_preview)

Chunks created: 4
Max characters per chunk: 1000

First chunk preview:
## About Me

# Ashish Subedi

I'm a fullstack developer from Nepal who builds end-to-end web applications — from pixel-perfect UIs to robust backend APIs. I work across the full stack with React , Next.js , NestJS , and TypeScript , backed by PostgreSQL for reliable, scalable data storage.

I've shi


## 5. Embed & Store in Chroma

`GoogleGenerativeAIEmbeddings` turns each chunk into a vector, and `Chroma.from_documents` stores those vectors in-memory for fast similarity search.

In [32]:
embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)
vectorstore = Chroma.from_documents(chunks, embeddings)

chunk_count = len(chunks)
print("Vector store built.")
print("Chunks stored:", chunk_count)

Vector store built.
Chunks stored: 4


## 6. Set Up the Retriever

The retriever searches Chroma for the `DEFAULT_TOP_K` chunks most similar to the question.

In [33]:
question = "What is this page about?"

retriever = vectorstore.as_retriever(search_kwargs={"k": DEFAULT_TOP_K})

## 7. Define the Structured Answer Schema

A small Pydantic model that Gemini fills in directly: `answer` holds the response text, and `is_answer_in_context` is a boolean flag for whether the context actually contained the answer.

In [34]:
class RagAnswer(BaseModel):
    """Structured answer Gemini must return."""

    answer: str = Field(
        description="The answer to the question, written using ONLY the given context. "
        "Empty string if the context does not contain the answer."
    )
    is_answer_in_context: bool = Field(
        description="True if the context contains enough information to answer the "
        "question, False otherwise."
    )

## 8. Build the Chat Prompt 

A **system** message sets the ground rules — answer only from `{context}`, and fill in `is_answer_in_context` honestly. A **human** message carries the actual question (`{input}`).

In [35]:
SYSTEM_TEMPLATE = """You are a helpful assistant that answers questions using ONLY the context below.

Rules:
- Write the answer strictly from the context. Do not use outside knowledge.
- If the context does not contain the answer, set is_answer_in_context to false and leave answer empty.

Context:
{context}"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_TEMPLATE),
        ("human", "{input}"),
    ]
)

## 9. Build the Retrieval Chain

`llm.with_structured_output(RagAnswer)` makes Gemini return a `RagAnswer` object. `create_stuff_documents_chain` is given `RunnablePassthrough` as its output parser, so that object passes straight through.

In [36]:
llm = ChatGoogleGenerativeAI(model=CHAT_MODEL, temperature=0)
structured_llm = llm.with_structured_output(RagAnswer)

combine_docs_chain = create_stuff_documents_chain(
    structured_llm, prompt, output_parser=RunnablePassthrough()
)
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

## 10. Ask the Question

In [37]:
response = rag_chain.invoke({"input": question})

structured_answer = response["answer"]
context_docs = response.get("context", [])

if structured_answer.is_answer_in_context:
    answer = structured_answer.answer.strip()
else:
    answer = FALLBACK_ANSWER

print("Question:", question)
print()
print("Answer:", answer)
print("Grounded in context:", structured_answer.is_answer_in_context)

Question: What is this page about?

Answer: This page is about Ashish Subedi, a fullstack developer from Nepal. It details his skills, technologies used (React, Next.js, NestJS, TypeScript, PostgreSQL), experience in building web applications and working as a freelance developer, and his contact location.
Grounded in context: True


## 11. Inspect the Retrieved Evidence

These are the exact chunks the retriever picked and handed to Gemini.

In [38]:
for index, doc in enumerate(context_docs, start=1):
    source = doc.metadata.get("source", "Unknown source")
    chunk_number = doc.metadata.get("chunk_number", "?")

    print("Evidence chunk:", index)
    print("Source:", source)
    print("Chunk number:", chunk_number)
    print(doc.page_content[:400])
    print()

Evidence chunk: 1
Source: https://www.ashishsubedi.com.np/
Chunk number: 1
## About Me

# Ashish Subedi

I'm a fullstack developer from Nepal who builds end-to-end web applications — from pixel-perfect UIs to robust backend APIs. I work across the full stack with React , Next.js , NestJS , and TypeScript , backed by PostgreSQL for reliable, scalable data storage.

I've shipped products at startups and as a freelancer — designing schemas, building REST APIs, and crafting 

Evidence chunk: 2
Source: https://www.ashishsubedi.com.np/
Chunk number: 1
## About Me

# Ashish Subedi

I'm a fullstack developer from Nepal who builds end-to-end web applications — from pixel-perfect UIs to robust backend APIs. I work across the full stack with React , Next.js , NestJS , and TypeScript , backed by PostgreSQL for reliable, scalable data storage.

I've shipped products at startups and as a freelancer — designing schemas, building REST APIs, and crafting 

Evidence chunk: 3
Source: https://www.ashishsu